#**Packages**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve_triangular

#**Necessary Utility Functions**

In [ ]:
def euclidean_norm(vector):
    sum_of_squares = sum(x**2 for x in vector)
    return sum_of_squares ** 0.5


def induced_A_norm(A, vector):
    A_vec = A * vector
    inner = np.dot(vector,A_vec)
    return inner ** 0.5


def generate_random_vector(a,b,n):
    vector = np.random.uniform(a,b,n)
    return vector


def plot_iterations(rich_list, sd_list, cg_list):

    iterations = range(len(rich_list))

    # Create a single plot
    plt.figure(figsize=(10, 6))

    # Plot Richardson's iterations
    plt.plot(iterations, rich_list, label="Richardson's Method", marker='o', color='blue', linestyle='-')

    # Plot Steepest Descent iterations
    plt.plot(iterations, sd_list, label='Steepest Descent', marker='s', color='green', linestyle='--')

    # Plot Conjugate Gradient iterations
    plt.plot(iterations, cg_list, label='Conjugate Gradient', marker='^', color='red', linestyle='-.')

    plt.title("Comparison of Iteration Counts for Different Methods", fontsize=14)
    plt.xlabel("Test Case", fontsize=12)
    plt.ylabel("Number of Iterations", fontsize=12)
    plt.legend(fontsize=12)
    plt.grid(alpha=0.4)
    plt.tight_layout()
    plt.savefig("Iterations_PT5_1000")
    plt.show()

#**Part 1 Algorithms**

In [ ]:
def richardson(Lambda, b_tilde, x0, x_tilde, epsilon = 1e-6, max_iter = 1000):

    #Parameters:
        #Lambda (np array): Diagonal matrix with eigenvalues stored as a vector
        #b_tilde (np array): Righthand side vector
        #x0 (np array): Initial guess of the solution
        #x_tilde (np array): True solution vector
        #epsilon (scalar): Convergence criterion for the residual norm
        #max_iter (scalar): Maximum number of iterations

    alpha = 2 / (np.min(Lambda) + np.max(Lambda))    #Set optimal alpha
    x = x0    #Initialize x to x0
    r = b_tilde - (Lambda * x)    #Initialize residual
    r0_norm = euclidean_norm(r)    #Compute the initial residual norm
    iteration = 0

    x_true = x_tilde
    error = euclidean_norm(x - x_true)    #Initializes error
    error_list = []

    while iteration < max_iter:
        x_next = x + (alpha * r)    #Update x
        error_next = euclidean_norm(x_next - x_true)
        rel_error = error_next / error    #Computes relative error at each iteration
        error_list.append(rel_error)

        r_next = r - (alpha * Lambda * r)    #Update residual
        r_next_norm = euclidean_norm(r_next)    #Compute current convergence level

        if r_next_norm / r0_norm < epsilon:
            return x_next, iteration+1, error_list    #Stops the iterative method if convergence criterion is met

        x = x_next    #Updates
        error = error_next
        r = r_next
        iteration = iteration + 1

    return x, iteration, error_list

In [ ]:
def steepest(Lambda, b_tilde, x0, x_tilde, epsilon = 1e-6, max_iter = 1000):

    #Parameters:
        #Lambda (np array): Diagonal matrix with eigenvalues stored as a vector
        #b_tilde (np array): Righthand side vector
        #x0 (np array): Initial guess of the solution
        #x_tilde (np array): true solution vector
        #epsilon (scalar): Convergence criterion for the residual norm
        #max_iter (scalar): Maximum number of iterations

    x = x0    #Initialize x to x0
    r = b_tilde - (Lambda * x)    #Initialize residual
    r0_norm = euclidean_norm(r)    #Compute the initial residual norm
    v = Lambda * r    #Initialize v
    iteration = 0

    x_true = x_tilde
    error = induced_A_norm(Lambda, x - x_true)    #Initializes error
    error_list = []

    while iteration < max_iter:
        alpha = (np.dot(r,r)) / (np.dot(r,v))    #Set optimal alpha at each step
        x_next = x + (alpha * r)    #Update x
        error_next = induced_A_norm(Lambda, x_next - x_true)
        rel_error = error_next / error    #Computes relative error at each iteration
        error_list.append(rel_error)

        r_next = r - (alpha * v)    #Update residual
        r_next_norm = euclidean_norm(r_next)    #Compute current convergence level

        if r_next_norm / r0_norm < epsilon:
            return x_next, iteration+1, error_list    #Stops the iterative method if convergence criterion is met

        v_next = Lambda * r_next    #Updates
        x = x_next
        error = error_next
        r = r_next
        v = v_next
        iteration = iteration + 1

    return x, iteration, error_list

In [ ]:
def conj_grad(Lambda, b_tilde, x0, x_tilde, epsilon = 1e-6, max_iter = 1000):

    #Parameters:
        #Lambda (np array): Diagonal matrix with eigenvalues stored as a vector
        #b_tilde (np array): Righthand side vector
        #x0 (np array): Initial guess of the solution
        #x_tilde (np array): true solution vector
        #epsilon (scalar): Convergence criterion for the residual norm
        #max_iter (scalar): Maximum number of iterations

    x = x0    #Initialize x to x0
    r = b_tilde - (Lambda * x)    #Initialize residual
    r0_norm = euclidean_norm(r)    #Compute the initial residual norm
    d = r    #Initialize d
    sigma = np.dot(r,r)    #Initialize sigma
    iteration = 0

    x_true = x_tilde
    error0 = induced_A_norm(Lambda, x - x_true)    #Initializes error
    error_list = []

    while iteration < max_iter:
        v = Lambda * d    #Compute v
        mu = np.dot(d,v)    #Compute mu
        alpha = sigma / mu    ##Set optimal alpha at each step
        x_next = x + (alpha * d)    #Update x
        error_next = induced_A_norm(Lambda, x_next - x_true)
        rel_error = error_next / error0    #Computes relative error at each iteration
        error_list.append(rel_error)

        r_next = r - (alpha * v)    #Update the residual
        r_next_norm = euclidean_norm(r_next)    #Compute current convergence level

        if r_next_norm / r0_norm < epsilon:
            return x_next, iteration+1, error_list    #Stops the iterative method if convergence criterion is met

        sigma_next = np.dot(r_next,r_next)    #Updates
        beta = sigma_next / sigma
        d_next = r_next + (beta * d)
        x = x_next
        r = r_next
        sigma = sigma_next
        d = d_next
        iteration = iteration + 1

    return x, iteration, error_list

#**Part 1 Tester**

In [ ]:
def main():

    problem_type = int(input("Please select which problem type you would like to analyze: \n "
                         "1. All n eigenvalues the same \n "
                         "2. k distinct eigenvalues with multiplicities chosen deterministically \n "
                         "3. k distinct eigenvalues with random distributions around each distinct eigenvalue \n "
                         "4. Eigenvalues chosen from a uniform distribution \n "
                         "5. Eigenvalues chosen from a normal distribution \n "
                         "Choice: "))


    n = 1000  #Default size of Lambda (can be adjusted)
    k = 90 #Default number of distinct eigenvalues for problem types 2 and 3
    lambda_min = 10  #Minimum eigenvalue for problem types 4 and 5
    lambda_max = 100  #Maximum eigenvalue for problem types 4 and 5

    if problem_type == 1:  #All n eigenvalues the same
        Lambda = np.full(n, 10)
        print(Lambda)

    elif problem_type == 2:  #k distinct eigenvalues with deterministic multiplicities
        distinct_values = np.linspace(lambda_min, lambda_max, k)
        multiplicities = [n // k] * (k - 1) + [n - (k - 1) * (n // k)]
        Lambda = np.hstack([np.full(m, val) for m, val in zip(multiplicities, distinct_values)])
        #np.random.shuffle(Lambda)  #Shuffle for non-ordered distribution
        print(Lambda)

    elif problem_type == 3:  # kdistinct eigenvalues with "clouds" around each
        distinct_means = np.linspace(lambda_min, lambda_max, k)
        multiplicities = [n // k] * (k - 1) + [n - (k - 1) * (n // k)]
        Lambda = np.hstack([
            np.random.normal(loc=mean, scale=(lambda_max - lambda_min) / (6 * k), size=m)
            for mean, m in zip(distinct_means, multiplicities)
        ])
        Lambda = np.absolute(Lambda)    #Eliminates negative values
        print(Lambda)

    elif problem_type == 4:  #Eigenvalues from uniform distribution
        remaining_values = np.random.uniform(low=lambda_min, high=lambda_max, size=n - 2)
        Lambda = np.hstack([lambda_min, lambda_max, remaining_values])
        print(Lambda)

    elif problem_type == 5:  #Eigenvalues from normal distribution
        remaining_values = np.random.normal(
            loc=(lambda_min + lambda_max) / 2,
            scale=(lambda_max - lambda_min) / 6,
            size=n - 2
        )
        remaining_values = np.clip(remaining_values, lambda_min, lambda_max)
        Lambda = np.hstack([lambda_min, lambda_max, remaining_values])
        print(Lambda)


    lambda_max = np.max(Lambda)
    lambda_min = np.min(Lambda)
    kappa = lambda_max / lambda_min
    print(kappa)
    error_bound = (kappa - 1)/(kappa + 1)
    print(error_bound)
    cg_alpha = ((kappa ** 0.5) - 1)/((kappa ** 0.5) + 1)


    figure, axis = plt.subplots(1, 2, figsize=(10, 4))  # Adjust figure size for better clarity
    axis[0].set_title("Richardson & Steepest Descent Error")
    axis[0].set_xlabel("Iteration")
    axis[0].set_ylabel("Error Ratio")
    axis[0].axhline(y=error_bound, color='r', linewidth=1.5, linestyle='-')
    axis[1].set_title("Conjugate Gradient Error")
    axis[1].set_xlabel("Iteration")
    axis[1].set_ylabel("Error Ratio")


    x_tilde = generate_random_vector(-50, 50, n)
    b_tilde = Lambda * x_tilde

    rich_iteration_list = []
    sd_iteration_list = []
    cg_iteration_list = []
    for i in range(10):
        x0 = generate_random_vector(-100, 100, n)

        rich_x, rich_iter_num, rich_error_list = richardson(Lambda, b_tilde, x0, x_tilde, epsilon = 1e-6, max_iter = 10000)
        steps1 = range(len(rich_error_list))
        print("RICH ERROR LIST: ", rich_error_list)
        axis[0].plot(steps1, rich_error_list, color = 'b')
        rich_iteration_list.append(rich_iter_num)

        sd_x, sd_iter_num, sd_error_list = steepest(Lambda, b_tilde, x0, x_tilde, epsilon = 1e-6, max_iter = 10000)
        steps2 = range(len(sd_error_list))
        print("STEPS: ", steps2)
        print("SD ERROR LIST: ", sd_error_list)
        axis[0].plot(steps2, sd_error_list, color = 'g')
        sd_iteration_list.append(sd_iter_num)

        cg_x, cg_iter_num, cg_error_list = conj_grad(Lambda, b_tilde, x0, x_tilde, epsilon = 1e-6, max_iter = 10000)
        steps3 = range(len(cg_error_list))

        bound_list = []
        for j in range(len(cg_error_list)):
            bound = 2 * (cg_alpha ** j)
            bound_list.append(bound)
        axis[1].plot(steps3, bound_list, color = 'r', linewidth=2.5, linestyle = '-')

        print("BOUND LIST: ", bound_list)
        print("CG ERROR LIST: ", cg_error_list)

        axis[1].plot(steps3, cg_error_list)
        cg_iteration_list.append(cg_iter_num)

    print(rich_iteration_list)
    print(sd_iteration_list)
    print(cg_iteration_list)

    plt.savefig("Errors_PT5_1000")
    plt.show

    plot_iterations(rich_iteration_list, sd_iteration_list, cg_iteration_list)



main()

#**Part 2 Algorithms**

In [ ]:
def Lv_solver(A, v):
    """
    Solves Lz = v for z, where L is a lower triangular matrix stored in A.

    Parameters:
    A (np array): A lower triangular matrix (2D np array).
    v (np array): The vector v in the equation Lz = v (1D np array).

    Returns:
    z (np array): Solution vector
    """
    #Find the dimension of v
    n = len(v)

    #Initialize the solution vector z as a NumPy array
    z = np.zeros(n)

    #Solve the first element of z
    z[0] = v[0] / A[0, 0]

    #Loop over the elements of z from 2 to n (starting at index 1 in Python)
    for i in range(1, n):
        #Use numpy slicing to calculate the sum of products A[i, :i] * z[:i]
        sum_prod = np.dot(A[i, :i], z[:i])

        #Solve for z[i] and divide by the diagonal element A[i, i]
        z[i] = (v[i] - sum_prod) / A[i, i]

    return z

In [ ]:
def Uv_solver(A, v):
    """
    Solves Uz = v for z, where U is an upper triangular matrix stored in A.

    Parameters:
    A (np array): An upper triangular matrix (2D np array).
    v (np array): The vector v in the equation Uz = v (1D np array).

    Returns:
    z (np array): Solution vector
    """
    #Find the dimension of v
    n = len(v)

    #Initialize the solution vector z as a NumPy array
    z = np.zeros(n)

    #Solve the last element of z
    z[-1] = v[-1] / A[-1, -1]

    #Loop over the elements of z in reverse order from n-2 to 0
    for i in range(n - 2, -1, -1):
        #Use numpy slicing to calculate the sum of products A[i, i+1:] * z[i+1:]
        sum_prod = np.dot(A[i, i+1:], z[i+1:])

        #Solve for z[i] and divide by the diagonal element A[i, i]
        z[i] = (v[i] - sum_prod) / A[i, i]

    return z

In [ ]:
def stationary_method(A, b, x0, x_tilde, method, epsilon=1e-6, max_iter=10000):

    #Parameters:
        #A (np array): Dense matrix
        #b (np array): Righthand side vector
        #x0 (np array): Initial guess of the solution
        #x_tilde (np array): True solution vector
        #method (int): Choice of stationary method
        #epsilon (scalar): Convergence criterion for the residual norm
        #max_iter (scalar): Maximum number of iterations

    L = np.tril(A)
    U = np.triu(A)

    if method == 1:
        #Initialize variables
        x = x0
        iteration = 0

        x_true = x_tilde
        true_norm = np.linalg.norm(x_true)

        while iteration < max_iter:
            #Perform the iteration
            r = b - (np.dot(A,x))  # Residual
            x_next = x + (1 / np.diag(A)) * r  # '*' is used because both variables are stored as vectors, so componentwise multiplication can be used
            error = np.linalg.norm(x_next - x_true)
            rel_error = error / true_norm

            if rel_error < epsilon:
                return x_next, iteration+1    #Stops the iterative method if convergence criterion is met

            x = x_next    #Updates
            iteration = iteration + 1

        return x, iteration

    elif method == 2:
        #Initialize variables
        x = x0
        iteration = 0

        x_true = x_tilde
        true_norm = np.linalg.norm(x_true)

        while iteration < max_iter:
            #Perform the iteration
            r = b - (np.dot(A,x))  # Residual
            z = Lv_solver(L, r)
            x_next = x + z
            error = np.linalg.norm(x_next - x_true)
            rel_error = error / true_norm

            if rel_error < epsilon:
                return x_next, iteration+1    #Stops the iterative method if convergence criterion is met

            x = x_next    #Updates
            iteration = iteration + 1

        return x, iteration

    elif method == 3:
        #Initialize variables
        x = x0
        iteration = 0

        x_true = x_tilde
        true_norm = np.linalg.norm(x_true)

        while iteration < max_iter:
            #Perform the iteration
            r = b - (np.dot(A,x))  #Residual
            z1 = Lv_solver(L, r)
            z2 = np.diag(A) * z1    # '*' is used because both variables are stored as vectors, so componentwise multiplication can be used
            z3 = Uv_solver(U, z2)
            x_next = x + z3
            error = np.linalg.norm(x_next - x_true)
            rel_error = error / true_norm

            if rel_error < epsilon:
                return x_next, iteration+1    #Stops the iterative method if convergence criterion is met

            x = x_next    #Updates
            iteration = iteration + 1

        return x, iteration

In [ ]:
def compute_G(A, n, method):
    D = np.diag(np.diag(A))
    L = np.tril(A)
    U = np.triu(A)

    if method == 1:
        G = np.identity(n) - (np.dot((1/np.diag(A)),A))
    elif method == 2:
        B = solve_triangular(L, A, lower=True)
        G = np.identity(n) - B
    elif method == 3:
        B = solve_triangular(L, A, lower=True)
        B2 = np.dot(D, B)
        B3 = solve_triangular(U, B2, lower=False)
        G = np.identity(n) - B3
    return G


def info_about_G(A, n, method):
    G = compute_G(A, n, method)
    print("G: ", G)
    eigs = np.linalg.eigvals(G)
    abs_eigs = np.absolute(eigs)
    spectral_radius = np.max(abs_eigs)
    matrix_norm = np.linalg.norm(G, 2)
    return spectral_radius, matrix_norm

#**Part 2 Tester**

In [ ]:
A0 = np.array([[3,7,-1],
               [7,4,1],
               [-1,1,2]])

A1 = np.array([[3,0,4],
               [7,4,2],
               [-1,-1,2]])

A2 = np.array([[-3,3,-6],
               [-4,7,-8],
               [5,7,-9]])

A3 = np.array([[4,1,1],
               [2,-9,0],
               [0,-8,-6]])

A4 = np.array([[7,6,9],
               [4,5,-4],
               [-7,-3,8]])

A5 = np.array([[6,-2,0],
               [-1,2,-1],
               [0,-6/5,1]])

A6 = np.array([[5,-1,0],
               [-1,2,-1],
               [0,-3/2,1]])

A7 = np.array([[4,-1,0,0,0,0,0],
               [-1,4,-1,0,0,0,0],
               [0,-1,4,-1,0,0,0],
               [0,0,-1,4,-1,0,0],
               [0,0,0,-1,4,-1,0],
               [0,0,0,0,-1,4,-1],
               [0,0,0,0,0,-1,4]])

A8 = np.array([[2,-1,0,0,0,0,0],
               [-1,2,-1,0,0,0,0],
               [0,-1,2,-1,0,0,0],
               [0,0,-1,2,-1,0,0],
               [0,0,0,-1,2,-1,0],
               [0,0,0,0,-1,2,-1],
               [0,0,0,0,0,-1,2]])



def main():

    method = int(input("Please select which stationary method you would like to use: \n "
                         "1. Jacobi \n "
                         "2. Forward Gauss-Seidel \n "
                         "3. Symmetric Gauss-Seidel \n "
                         "Choice: "))

    matrix = int(input("Please select which number matrix you would like to use (0 through 9): \n"
                       "Choice: "))

    if matrix == 0:
        A = A0
        n = 3
    elif matrix == 1:
        A = A1
        n = 3
    elif matrix == 2:
        A = A2
        n = 3
    elif matrix == 3:
        A = A3
        n = 3
    elif matrix == 4:
        A = A4
        n = 3
    elif matrix == 5:
        A = A5
        n = 3
    elif matrix == 6:
        A = A6
        n = 3
    elif matrix == 7:
        A = A7
        n = 7
    elif matrix == 8:
        A = A8
        n = 7


    spectral_radius, matrix_norm = info_about_G(A, n, method)
    print("Spectral Radius of G: ", spectral_radius)
    print("2-norm of G: ", matrix_norm)


    x_tilde = np.ones(n)
    b = np.dot(A,x_tilde)
    x0 = np.zeros(n)

    new_x, iteration_num = stationary_method(A, b, x0, x_tilde, method, epsilon=1e-6, max_iter=10000)
    print("Iterations until Convergence: ", iteration_num)




    j_iteration_list = []
    gs_iteration_list = []
    sgs_iteration_list = []

    for i in range(20):
        x0 = generate_random_vector(-50, 50, n)
        x_tilde = generate_random_vector(-50, 50, n)
        b = np.dot(A,x_tilde)

        j_x, j_iter_num = stationary_method(A, b, x0, x_tilde, 1, epsilon=1e-6, max_iter=250)
        j_iteration_list.append(j_iter_num)

        gs_x, gs_iter_num = stationary_method(A, b, x0, x_tilde, 2, epsilon=1e-6, max_iter=250)
        gs_iteration_list.append(gs_iter_num)

        sgs_x, sgs_iter_num = stationary_method(A, b, x0, x_tilde, 3, epsilon=1e-6, max_iter=250)
        sgs_iteration_list.append(sgs_iter_num)


    iterations = range(len(j_iteration_list))

    # Create a single plot
    plt.figure(figsize=(10, 6))

    # Plot Jacobi iterations
    plt.plot(iterations, j_iteration_list, label="Jacobi", marker='o', color='blue', linestyle='-')

    # Plot Gauss-Seidel iterations
    plt.plot(iterations, gs_iteration_list, label='Gauss-Seidel', marker='s', color='green', linestyle='--')

    # Plot Symmetric Gauss-Seidel iterations
    plt.plot(iterations, sgs_iteration_list, label='Symmetric Gauss-Seidel', marker='^', color='red', linestyle='-.')

    plt.title("Comparison of Iteration Counts for Different Methods for Matrix A" + str(matrix), fontsize=14)
    plt.xlabel("Test Case", fontsize=12)
    plt.ylabel("Number of Iterations", fontsize=12)
    plt.legend(fontsize=12)
    plt.grid(alpha=0.4)
    plt.tight_layout()
    plt.savefig("P2_Iterations_A8")
    plt.show()



main()

#**Part 3 Algorithms**

In [ ]:
def generate_sparse_spd_matrix(a, b, n, sparsity, multiplier):
    """
    Generates a random sparse SPD nxn matrix with entries between a and b.
    Sparsity can be controlled and the diagonal is boosted to ensure positive definiteness.
    """
    A = np.zeros((n,n))    #Initialize Matrix A

    for i in range(n):
        nonzeros = max(1, int(sparsity * i))    #Limits the number of nonzero elements in the i'th row

        if nonzeros > i:
            nonzeros = i

        ind = np.random.choice(i, size=nonzeros, replace=False)

        A[i, ind] = np.random.uniform(a, b, size=nonzeros)

        for j in ind:
            A[j, i] = A[i, j]    #Making A symmetric

    diagonals = generate_random_vector(a,b,n)
    for i in range(n):
        A[i, i] = diagonals[i]    #Sets diagonal elements to random values

    #Boost the diagonal of A to ensure positive definiteness
    for i in range(n):
        off_diag = np.sum(np.abs(A[i, :])) - np.abs(A[i, i])
        boosted = off_diag * multiplier
        if boosted > A[i,i]:
            A[i,i] = boosted

    return A

In [ ]:
def lower_tri_csr(A):
    """
    Takes in a sparse matrix A and stores the nonzero elements in the lower triangular part in CSR format.
    """
    n = len(A)

    aa = []    #array of values

    ja = []    #array of column indices

    ia = np.zeros(n+1, dtype=int)    #list of row pointers

    count = 0

    ia[0] = 0

    for i in range(n):
        for k in range(i):
            if A[i, k] != 0:
                aa.append(A[i,k])    #append element to list of values
                ja.append(k)    #append the column index
                count += 1
        ia[i + 1] = count    #update the list of row pointers

    aa = np.array(aa)
    ja = np.array(ja)

    return aa, ja, ia

In [ ]:
def sparse_matrix_vector_prod(aa, ia, ja, x, D):
    """
    Takes in the three components of the CSR storage, a vector x to multiply and the diagonal of A.
    """
    n = len(x)

    z = np.zeros(n)    #Initialization

    for i in range(n):
        z[i] = z[i] + (D[i] * x[i])

        start = ia[i]
        end = ia[i + 1]    #note the starting and ending indices for the i'th row

        for k in range(start, end):
            col_index = ja[k]

            val = aa[k]

            z[i] = z[i] + (val * x[col_index])    #update from lower triangular part

            z[col_index] = z[col_index]+ (val * x[i])    #update from upper triangular part

    return z

In [ ]:
def sparse_lower_solve(aa, ia, ja, b, D):
    """
    Takes in the three components of the CSR storage, a righthand side vector b and the diagonal of A.
    """
    n = len(b)

    z = np.zeros(n)

    z[0] = b[0] / D[0]    #fist element of z

    for i in range(1, n):
        start = ia[i]    #starting index
        end = ia[i + 1]    #ending index

        mid_sum = np.dot(aa[start:end], z[ja[start:end]])

        z[i] = (b[i] - mid_sum) / D[i]

    return z